# Backdoor Toolbox — Experiment Runner
Execute cells in order. Only **Cell 3** needs to be edited to select attacks.

## Cell 1 — Environment Setup

In [ ]:
import os, sys

# ── Detect platform ──────────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules

REPO_URL  = 'https://github.com/your-username/backdoor-toolbox.git'  # ← change this
REPO_DIR  = 'backdoor-toolbox'

if ON_KAGGLE:
    ROOT = f'/kaggle/working/{REPO_DIR}'
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = f'/content/drive/MyDrive/{REPO_DIR}'
else:
    ROOT = os.path.abspath(REPO_DIR)

# ── Clone repo (skip if already exists) ──────────────────────────────────────
if not os.path.isdir(ROOT):
    os.system(f'git clone {REPO_URL} {ROOT}')

os.chdir(ROOT)
print(f'Working directory: {os.getcwd()}')

## Cell 2 — Import

In [ ]:
import tomllib
from generator import phase1, phase2, phase3, phase_other_attack

with open('config.toml', 'rb') as f:
    cfg = tomllib.load(f)['Trainer']

print('Config loaded:')
for k, v in cfg.items():
    if not isinstance(v, dict):
        print(f'  {k} = {v}')

## Cell 3 — Select Attacks

Uncomment the attacks you want to run.

**Poisoning attacks** (use the standard pipeline):
```
none, badnet, blend, trojan, SIG, dynamic, ISSBA,
WaNet, TaCT, refool, adaptive_blend, adaptive_patch, adaptive_k_way,
badnet_all_to_all, SleeperAgent, clean_label
```
**Other attacks** (run via `other_attack.py` internally):
```
bpp, trojannn, BadEncoder, SRA, WB
```

In [ ]:
POISON_TYPES = [
    # ── Poisoning attacks ────────────────────────────────────────
    # 'none',
    # 'badnet',
    # 'blend',
    # 'trojan',
    # 'SIG',
      'dynamic',
    # 'ISSBA',
      'WaNet',
    # 'TaCT',
    # 'refool',
      'adaptive_blend',
      'adaptive_patch',
    # 'adaptive_k_way',
    # 'badnet_all_to_all',
    # 'SleeperAgent',
    # 'clean_label',

    # ── Other attacks (run via other_attack.py) ──────────────────
    # 'bpp',
    # 'trojannn',
    # 'BadEncoder',
    # 'SRA',
    # 'WB',
]

OTHER_ATTACKS = {'bpp', 'trojannn', 'BadEncoder', 'SRA', 'WB'}

print(f'Selected attacks: {POISON_TYPES}')

## Cell 4 — Run

In [ ]:
import time

def run_process(cfg, poison_type):
    dataset     = cfg['dataset']
    data_rate   = cfg['data_rate']
    poison_rate = cfg['poison_rate']
    num_models  = cfg['num_models']
    train_source= cfg['train_source']
    clean_budget= cfg['clean_budget']
    cover_rate  = None

    # Phase 1
    phase1(dataset, clean_budget)

    # Phase 2 / other attack
    if poison_type in OTHER_ATTACKS:
        phase_other_attack(
            dataset, data_rate, poison_type, poison_rate,
            num_models, train_source, clean_budget, cover_rate,
        )
        return  # other_attack handles training internally

    if poison_type in ('WaNet', 'TaCT', 'adaptive_blend', 'adaptive_patch'):
        cover_rate = cfg[poison_type]['cover_rate']

    phase2(
        dataset, data_rate, poison_type, poison_rate,
        train_source, clean_budget, cover_rate,
    )

    # Phase 3
    phase3(
        dataset, data_rate, poison_type, poison_rate,
        num_models, train_source, clean_budget, cover_rate,
    )


total = len(POISON_TYPES)
for idx, poison_type in enumerate(POISON_TYPES, 1):
    print(f'\n==============================')
    print(f'[{idx}/{total}] poison_type = {poison_type}')
    print(f'==============================')
    t0 = time.time()
    run_process(cfg, poison_type)
    elapsed = time.time() - t0
    print(f'[{idx}/{total}] {poison_type} done in {elapsed/60:.1f} min')

print('\nAll experiments completed.')